# 🔬 nautilus-lab: Повний інтерактивний посібник з кількісних досліджень

Цей зошит демонструє стандартизований цикл розробки, перевірки та аудиту алгоритмічних стратегій на базі **NautilusTrader** у репозиторії `nautilus-lab`.

### Архітектурні принципи:
1. **Research-First**: Усі дослідження проходять строгий аудит на перенавчання перед будь-яким розгортанням.
2. **Zero Lookahead**: Сигнали формуються виключно на закритих барах (`ts_event`), виконання моделюється із затримкою та реалістичними комісіями (Maker 0.02%, Taker 0.05%, спред/прослизання).
3. **In-Sample для вибору, Out-of-Sample для оцінки**: Оптимізовані параметри ніколи не оцінюються на навчальній вибірці.
4. **Purged Embargo**: Між IS та OOS встановлюється часовий буфер (embargo), щоб запобігти витоку інформації через серійну кореляцію.
5. **Fail-Closed**: Жодна стратегія не переходить до торгівлі без проходження статистичних воріт (PBO < 0.5, OOS Sharpe > 0, позитивний результат на стрес-слайсах).

## 1. Ініціалізація оточення та імпорти

Підключаємо модулі системи: конфігурацію, доменні моделі барів, індикатори мікроструктури, Nautilus-двигунець та валідатори.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, UTC, timedelta
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Переконуємось, що src доступний у sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from nautilus_lab.infrastructure.config import settings
from nautilus_lab.domain.bars import Bar, BarOrigin
from nautilus_lab.domain.windows import UtcWindow
from nautilus_lab.domain.regime import RobotName
from nautilus_lab.domain.metrics import BacktestMetrics
from nautilus_lab.application.run_walk_forward import walk_forward_request, walk_forward_use_case
from nautilus_lab.application.run_overfitting_audit import pbo_request, pbo_use_case

cfg = settings()
print(f"✓ Конфігурація завантажена: інструмент={cfg.instrument_id}, каталог={cfg.catalog_path}")
print(f"✓ Режим торгівлі: {cfg.trading_mode} (live fail-closed: {cfg.live_trading_disabled})")

## 2. Робота з даними: Parquet-каталог або синтетичні бари

У `nautilus-lab` дані зберігаються у високопродуктивному каталозі Apache Parquet (`catalog/`).
Для швидкої розробки та перевірки логіки доступний синтетичний генератор барів із фіксованим seed (`--synthetic`).

In [ ]:
# Завантаження даних із каталогу або перевірка його цілісності
from nautilus_trader.persistence.catalog import ParquetDataCatalog

catalog_dir = project_root / cfg.catalog_path
print(f"Шлях до каталогу: {catalog_dir}")

catalog_exists = catalog_dir.exists() and any(catalog_dir.iterdir())
if catalog_exists:
    try:
        catalog = ParquetDataCatalog(str(catalog_dir))
        bar_types = [cfg.bar_type_spec]
        bars = catalog.bars(bar_types=bar_types)
        print(f"✓ У каталозі знайдено {len(bars)} барів для {cfg.bar_type_spec}")
        if bars:
            print(f"  Діапазон: від {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')} до {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")
    except Exception as e:
        print(f"ℹ Каталог порожній або потребує оновлення ({e}). Можна використовувати --synthetic бари.")
else:
    print("ℹ Каталог ще не заповнено. Завантажте історію через:")
    print("   uv run lab ingest --start 2025-01-01 --symbols ETHUSDT,BTCUSDT")

## 3. Sanity Check: Перевірка даних на дублікати та розриви

> ⚠️ **Критичне правило:** Часові мітки барів мають строго монотонно зростати (`ts_event[i] > ts_event[i-1]`).
> Будь-які дублікати ламають сортування подій у симуляторі Nautilus.

In [ ]:
# Перевірка монотонності та відсутності дублікатів
def verify_bars_integrity(timestamps_ns: list[int]) -> dict:
    if not timestamps_ns:
        return {"status": "empty", "count": 0, "duplicates": 0, "monotonic": True}
    
    diffs = np.diff(timestamps_ns)
    duplicates = int(np.sum(diffs == 0))
    backwards = int(np.sum(diffs < 0))
    monotonic = duplicates == 0 and backwards == 0
    
    return {
        "count": len(timestamps_ns),
        "duplicates": duplicates,
        "backwards_steps": backwards,
        "monotonic": monotonic,
        "status": "OK" if monotonic else "CORRUPTED"
    }

print("✓ Функція перевірки цілісності готова.")

## 4. Фічі та фільтри мікроструктури: VPIN, Hawkes, Donchian, ATR

У `src/nautilus_lab/domain/` реалізовано доменні розрахунки:
- **VPIN (Volume-Synchronized Probability of Toxicity)**: Оцінка токсичного потоку та ризику несприятливого вибору (adverse selection).
- **Hawkes Process**: Кластеризація волатильності та інтенсивність потоку угод.
- **Donchian & ATR**: Фільтри динамічної волатильності та рівні пробою каналів.

In [ ]:
from nautilus_lab.domain.atr import atr_series
from nautilus_lab.domain.vpin import bar_vpin
from nautilus_lab.domain.donchian import donchian_channel

# Демонстрація розрахунку на згенерованій серії цін
np.random.seed(42)
n_bars = 200
close_prices = 3000.0 * np.exp(np.cumsum(np.random.normal(0.0002, 0.008, n_bars)))
high_prices = close_prices * (1 + np.abs(np.random.normal(0, 0.004, n_bars)))
low_prices = close_prices * (1 - np.abs(np.random.normal(0, 0.004, n_bars)))
volumes = np.random.lognormal(mean=5.0, sigma=0.8, size=n_bars)

atr_vals = atr_series(high_prices, low_prices, close_prices, period=14)
vpin_vals = bar_vpin(close_prices, volumes, window=30)
upper_d, lower_d = donchian_channel(high_prices, low_prices, period=20)

print(f"Згенеровано {n_bars} тестових барів.")
print(f"Останній ATR(14): {atr_vals[-1]:.2f}")
print(f"Останній VPIN: {vpin_vals[-1]:.4f} (індикатор токсичності потоку)")
print(f"Donchian(20): Upper={upper_d[-1]:.2f}, Lower={lower_d[-1]:.2f}")

## 5. Запуск Walk-Forward бектесту через NautilusTrader

Основний метод перевірки стратегій у репозиторії: **Walk-Forward Analysis**.
- **In-Sample (IS)**: 70% історії для калібрування параметрів.
- **Embargo Buffer**: 50 барів паузи для запобігання витоку інформації.
- **Out-of-Sample (OOS)**: 30% історії — єдиний інтервал, за яким приймається рішення!

In [ ]:
# Запуск чесного Walk-Forward тесту для обраного робота
robot_choice = RobotName.REGIME  # або EMA, PAIRS, VPIN_MOMENTUM, FORMULAIC_LGBM

req = walk_forward_request(
    cfg,
    robot=robot_choice,
    source=BarOrigin.SYNTHETIC,  # BarOrigin.CATALOG для реальних даних
    bar_count=3000,
    in_sample_fraction=0.7,
    embargo_bars=50,
    folds=1,
)

use_case = walk_forward_use_case(cfg)
wf_result = use_case.execute(req)

print("=== РЕЗУЛЬТАТИ WALK-FORWARD ===")
print(f"Параметри, обрані на In-Sample: {wf_result.selected_params}")
print(f"In-Sample баланс:  {wf_result.in_sample.ending_balance:.2f} (fills={wf_result.in_sample.fill_count})")
print(f"★ Out-of-Sample баланс (ЗВІТНИЙ): {wf_result.out_of_sample.ending_balance:.2f} (fills={wf_result.out_of_sample.fill_count})")
print(f"OOS PnL: {wf_result.out_of_sample.pnl_percent:+.2f}%")

## 6. Багатовіконний Walk-Forward (K-Folds Cross-Validation)

Одне OOS-вікно може випадково опинитись у сприятливому тренді. Щоб довести наявність альфи, ми розбиваємо історію на $K$ фолдів (`--folds 4`).
Стратегія вважається надійною, якщо вона показує стабільний результат на більшості фолдів.

In [ ]:
multi_req = walk_forward_request(
    cfg,
    robot=robot_choice,
    source=BarOrigin.SYNTHETIC,
    bar_count=3000,
    folds=4,
)

multi_res = use_case.execute_multi(multi_req)

print("=== РЕЗУЛЬТАТИ 4-FOLD WALK-FORWARD ===")
print(f"Успішних фолдів: {multi_res.profitable_folds} / {len(multi_res.folds)}")
print(f"Середня OOS прибутковість: {multi_res.mean_oos_return:+.2%}")
print(f"Середній Buy&Hold ринку:   {multi_res.mean_buy_and_hold_return:+.2%}")
print(f"Всього OOS угод: {multi_res.total_oos_fills}")

for i, fold in enumerate(multi_res.folds, 1):
    print(f"  Fold {i}: OOS={fold.out_of_sample.pnl_percent:+.2f}% (fills={fold.out_of_sample.fill_count})")

## 7. Аудит на перенавчання: CSCV та PBO (Probability of Backtest Overfitting)

За методологією Bailey & López de Prado (2014):
- Ми розбиваємо часовий ряд на $N$ блоків (наприклад, 8).
- Формуємо всі симетричні комбінації трейн/тест (комбінаторний крос-валідатор CSCV).
- Розраховуємо **PBO (Probability of Backtest Overfitting)**:
  - $PBO < 0.25$: Відмінно, вибір на In-Sample стійко узагальнюється на Out-of-Sample.
  - $PBO \approx 0.50$: Підкидання монети (відсутність стійкого edge).
  - $PBO > 0.50$: **Перенавчання!** In-sample переможець системно зливає поза вибіркою.

In [ ]:
pbo_req_obj = pbo_request(
    cfg,
    robot=robot_choice,
    source=BarOrigin.SYNTHETIC,
    bar_count=3000,
    blocks=8,
)

pbo_runner = pbo_use_case(cfg)
pbo_audit = pbo_runner.execute(pbo_req_obj)

print("=== АУДИТ НА ПЕРЕНАВЧАННЯ (PBO / CSCV) ===")
print(f"Блоків історії: {pbo_audit.blocks}")
print(f"Кількість протестованих конфігурацій: {pbo_audit.configuration_count}")
print(f"Кількість симетричних сплітів: {pbo_audit.splits_count}")
print(f"★ PBO = {pbo_audit.pbo:.4f}")

if pbo_audit.pbo < 0.30:
    print("🟢 ВИСНОВОК: Стратегія успішно пройшла аудит! Selection generalises.")
elif pbo_audit.pbo < 0.50:
    print("🟡 ВИСНОВОК: Прийнятний рівень, але потрібні додаткові стрес-тести.")
else:
    print("🔴 ВИСНОВОК: ПЕРЕНАВЧАННЯ (Overfitted). Стратегію відхилено.")

## 8. Стрес-тестування на історичних шоках (Stress Slices)

Стратегія має бути протестована на найбільш руйнівних періодах крипторинку:
- `covid2020` (Березень 2020: паніка ліквідності, падіння -50% за день)
- `ftx2022` (Листопад 2022: крах біржі FTX, каскадні ліквідації)
- `etf2024` (Січень 2024: волатильність схвалення Spot ETF)

CLI-виклик для каталогу:
```bash
uv run lab research --robot regime --slice ftx2022 --catalog catalog
```

In [ ]:
from nautilus_lab.domain.stress_slices import NamedStressSlice

print("Доступні стрес-слайси:")
for s in NamedStressSlice:
    print(f"  • {s.value}")

## 9. Офлайн LLM-контур генерації альф (`lab propose`) та Журнал рішень

У `nautilus-lab` модель ШІ використовується **виключно офлайн**:
- LLM виступає генератором математичних гіпотез (`research/prompts/01-generate-alphas.md`).
- Кожна відповідь валідується за схемою Pydantic і записується як JSON у `research/hypotheses/`.
- Жоден LLM-код не йде в торгівлю без рев'ю людини та проходження тестів.
- Кожен результат тестування записується у `research/journal.md` та `research/journal.jsonl`.

In [ ]:
from nautilus_lab.application.journal import JournalRecord, record_journal_entry

# Приклад запису рішення в журнал
entry = JournalRecord(
    date=datetime.now(UTC).strftime("%Y-%m-%d"),
    subject="regime_walk_forward_demo",
    gates="purged walk-forward folds=4",
    oos=f"{multi_res.mean_oos_return:+.2%}",
    buy_and_hold=f"{multi_res.mean_buy_and_hold_return:+.2%}",
    decision="accepted" if multi_res.profitable_folds >= 3 else "rejected",
    reason=f"Profitable in {multi_res.profitable_folds}/4 folds, PBO={pbo_audit.pbo:.2f}",
)

print("Приклад згенерованого рядка для research/journal.md:")
print(f"| {entry.date} | {entry.subject} | {entry.gates} | {entry.oos} | {entry.buy_and_hold} | {entry.decision} | {entry.reason} |")

## 10. Інтерактивна візуалізація кривої капіталу (Tearsheet)

Будуємо криву капіталу (Equity Curve) та графік підводних просадок (Underwater Drawdown) за результатами симуляції.

In [ ]:
# Візуалізація результатів
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Equity Curve (Капітал)", "Underwater Drawdown (Просадка %)")
)

# Синтетичний приклад динаміки еквіті для графіка
steps = 100
initial_cap = 100_000.0
returns = np.random.normal(0.001, 0.01, steps)
equity_curve = initial_cap * np.cumprod(1 + returns)
peak = np.maximum.accumulate(equity_curve)
drawdown = (equity_curve - peak) / peak * 100

fig.add_trace(
    go.Scatter(y=equity_curve, mode="lines", name="Strategy Equity", line=dict(color="#00d4aa", width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(y=drawdown, mode="lines", name="Drawdown %", fill="tozeroy", line=dict(color="#ff4d4d", width=1)),
    row=2, col=1
)

fig.update_layout(
    template="plotly_dark",
    height=600,
    title="<b>Аналітичний звіт стратегії (Interactive Tearsheet)</b>",
    showlegend=True
)

fig.show()

## 11. Підсумковий чекліст готовності стратегії до Paper Trading

Перед тим, як перевести робота в режим paper trading (`lab paper`), пройдіть цей обов'язковий чекліст:

| # | Етап валідації | Критерій допуску | Статус |
|---|---|---|:---:|
| 1 | **Unit & Type Tests** | `uv run pytest` (100% pass) + `uv run mypy src tests` (0 errors) | [ ] |
| 2 | **Комісії та спред** | Maker 0.02%, Taker 0.05% враховано в симуляторі | [ ] |
| 3 | **Purged Walk-Forward** | OOS Sharpe > 0.5, середня OOS дохідність > Buy&Hold | [ ] |
| 4 | **Багатовіконність** | $\ge 75\%$ фолдів прибуткові (`--folds 4`) | [ ] |
| 5 | **Аудит перенавчання (PBO)** | $PBO < 0.40$ (за де Прадо) | [ ] |
| 6 | **Стрес-тести** | Відсутність маржин-колів на слайсах `covid2020`, `ftx2022` | [ ] |
| 7 | **Запис у журнал** | Гіпотеза та результат зафіксовані у `research/journal.md` | [ ] |

Якщо всі 7 пунктів виконані — запускаємо моніторинг:
```bash
uv run lab paper --robot regime
```